Process Helathy-Aging-KG: Harmonize NodeType; Change Node ids & Edge ids

In [1]:
import torch
from torch_geometric.data import HeteroData
import networkx as nx
from typing import Tuple, Dict, Any
from tqdm import tqdm
import copy

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from utils.graph_utils import load_graph, save_graph
from data_processing.patient_network_prep import convert_to_hetero_data, process_and_save

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
health_kg = load_graph("./data/KG/healthy_aging_reversed_remove_noncausal.pkl")

Loaded graph from ./data/KG/healthy_aging_reversed_remove_noncausal.pkl: 4161 nodes, 13792 edges


In [ ]:
def rename_node_edge_ids(kg):
    
    mapping = {}
    for node, data in kg.nodes(data=True):
        name = data.get('bel')
        mapping[node] = name
    
    # change node ids
    kg = nx.relabel_nodes(kg, mapping, copy=True)

    # change edge ids
    new_kg = nx.MultiDiGraph()
    new_kg.add_nodes_from(kg.nodes(data=True))
    for u,v,old_rel, data in kg.edges(data=True, keys=True):
        new_rel = data.get('type')
        new_kg.add_edge(u,v,new_rel, **data)
    
    return new_kg

In [13]:
new_health_kg = rename_node_edge_ids(health_kg)

In [10]:
node_types = set()
for node, data in new_health_kg.nodes(data=True):
    # print(node)
    # print(data)
    # print()
    node_type = data.get('type')
    node_types.add(node_type)
    #break
# for u,v,rel, data in new_health_kg.edges(data=True, keys=True):
    # print(u, rel, v)
    # print(data)
    # print()
    #break

In [11]:
node_types

{'Abundance',
 'Activity',
 'Biological_process',
 'Cell_secretion',
 'Cell_surface_expression',
 'Complex',
 'Composite',
 'Degradation',
 'Fragment',
 'From_location',
 'Fusion_protein',
 'Gene',
 'Location',
 'Pmod',
 'Products',
 'Protein',
 'Reactants',
 'Reaction',
 'Rna',
 'To_location',
 'Translocation',
 'Variant'}

In [12]:
import re


def sanitize_node_types(G):
    """
    Standardizes node types while preserving existing PascalCase names.
    - 'Cell_surface_expression' -> 'CellSurfaceExpression'
    - 'CellSurfaceExpression' -> 'CellSurfaceExpression' (UNTOUCHED)
    - 'biological_process' -> 'BiologicalProcess'
    """
    def fix_type_name(text):
        if not text:
            return "Unknown"
        
        # If there are underscores or spaces, we need to join them
        if '_' in text or ' ' in text:
            words = re.split(r'[_\s]+', str(text))
            # Capitalize each part and join: 'cell_surface' -> 'CellSurface'
            return ''.join(word[0].upper() + word[1:] if len(word) > 0 else '' for word in words)
        
        # If no delimiters, just ensure the very first letter is uppercase
        # but leave the rest of the string exactly as it is.
        return text[0].upper() + text[1:]

    type_changes = {}
    
    for node, attrs in G.nodes(data=True):
        old_type = attrs.get('type')
        if old_type:
            new_type = fix_type_name(old_type)
            if old_type != new_type:
                type_changes[old_type] = new_type
            attrs['type'] = new_type
            
    if type_changes:
        print("Sanitized Node Types (Smart Mapping):")
        for old, new in sorted(type_changes.items()):
            print(f"  {old} -> {new}")
            
    return G

In [15]:
new_health_kg = sanitize_node_types(new_health_kg)
nodeType = set()
for node, data in new_health_kg.nodes(data=True):
    nodeType.add(data.get('type'))
    #print('node type', data.get('type'))
print(nodeType)
save_graph(new_health_kg, './data/KG/healthy_aging_reversed_remove_noncausal.pkl')

{'ToLocation', 'Degradation', 'Pmod', 'Protein', 'Reaction', 'CellSurfaceExpression', 'Fragment', 'Composite', 'FromLocation', 'Reactants', 'BiologicalProcess', 'Abundance', 'FusionProtein', 'Complex', 'Products', 'Location', 'Activity', 'Translocation', 'Rna', 'CellSecretion', 'Variant', 'Gene'}
Saved graph to ./data/KG/healthy_aging_reversed_remove_noncausal.pkl: 4161 nodes, 13775 edges


In [16]:
# process other healthy-aging-kgs
kg0 = load_graph("./data/KG/healthy_aging_kg.pkl")
kg1 = load_graph("./data/KG/healthy_aging_remove_noncausal.pkl")

new_kg0 = rename_node_edge_ids(kg0)
new_kg0 = sanitize_node_types(new_kg0)
save_graph(new_kg0,"./data/KG/hamonized_healthy_aging_kg.pkl")

new_kg1 = rename_node_edge_ids(kg1)
new_kg1 = sanitize_node_types(new_kg1)
save_graph(new_kg1, "./data/KG/healthy_aging_remove_noncausal.pkl")

Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Loaded graph from ./data/KG/healthy_aging_remove_noncausal.pkl: 4161 nodes, 6896 edges
Sanitized Node Types (Smart Mapping):
  Biological_process -> BiologicalProcess
  Cell_secretion -> CellSecretion
  Cell_surface_expression -> CellSurfaceExpression
  From_location -> FromLocation
  Fusion_protein -> FusionProtein
  To_location -> ToLocation
Saved graph to ./data/KG/hamonized_healthy_aging_kg.pkl: 4161 nodes, 6956 edges
Sanitized Node Types (Smart Mapping):
  Biological_process -> BiologicalProcess
  Cell_secretion -> CellSecretion
  Cell_surface_expression -> CellSurfaceExpression
  From_location -> FromLocation
  Fusion_protein -> FusionProtein
  To_location -> ToLocation
Saved graph to ./data/KG/healthy_aging_remove_noncausal.pkl: 4161 nodes, 6896 edges
